# Data Cleaning - Maison_d'hôte Vente Marrakech
This notebook focuses on cleaning the maison_d'hôte vente data for Marrakech.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# File path
file_path = '../../data/marrakech_immo_vente_features/maison_d'hôte_vente.csv'

# Load data
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    display(df.head())
else:
    print(f"ERROR: File not found at {file_path}")
    print(f"Current working directory: {os.getcwd()}")

## 1. Initial Data Overview

In [ ]:
if 'df' in locals():
    print("Shape:", df.shape)
    df.info()
else:
    print("DataFrame 'df' not loaded. Please fix the file path above.")

## 2. Handling Missing Values

In [ ]:
if 'df' in locals():
    # Checking for null values
    null_counts = df.isnull().sum()
    print("Columns with null values before cleaning:")
    print(null_counts[null_counts > 0])
    
    # 1. Variables numériques : imputation par la médiane
    num_cols_to_fill = ['prix_num', 'surface_num', 'prix_m2', 'prix_m2_median_quartier']
    for col in num_cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
            
    # 2. Variables catégorielles : imputation par le mode (valeur la plus fréquente)
    cat_cols_mode = ['agence', 'type_bien', 'localisation']
    for col in cat_cols_mode:
        if col in df.columns:
            mode_val = df[col].mode()
            if not mode_val.empty:
                df[col] = df[col].fillna(mode_val[0])
            
    # 3. Champs texte : on met 'Non spécifié'
    text_cols = ['titre', 'description', 'prix', 'surface', 'url']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('Non spécifié')
    
    if 'maison_d_hôte' == 'terrain':
        # 4. Spécifique aux terrains : pas de chambres/sdb
        if 'chambres' in df.columns:
            df['chambres'] = df['chambres'].fillna('0')
        if 'salles_bain' in df.columns:
            df['salles_bain'] = df['salles_bain'].fillna(0.0)
        if 'surface_terrain' in df.columns and 'surface_num' in df.columns:
            df['surface_terrain'] = df['surface_terrain'].fillna(df['surface_num'])
    
    # id: on supprime les lignes sans ID car c'est crucial
    if 'id' in df.columns:
        df.dropna(subset=['id'], inplace=True)
        
    print("\nColumns with null values after cleaning:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Removing Duplicates

In [ ]:
if 'df' in locals():
    initial_size = len(df)
    df = df.drop_duplicates()
    print(f"Removed {initial_size - len(df)} duplicate rows.")

## 4. Cleaning Numerical Columns
Handling outliers for price and surface.

In [ ]:
if 'df' in locals():
    if 'prix_num' in df.columns:
        q_low = df['prix_num'].quantile(0.01)
        q_hi  = df['prix_num'].quantile(0.99)
        df = df[(df['prix_num'] <= q_hi) & (df['prix_num'] >= q_low)]
        print(f"Data points after price outlier removal: {len(df)}")
    
    if 'surface_num' in df.columns:
        q_low_surf = df['surface_num'].quantile(0.01)
        q_hi_surf  = df['surface_num'].quantile(0.99)
        df = df[(df['surface_num'] <= q_hi_surf) & (df['surface_num'] >= q_low_surf)]
        print(f"Data points after surface outlier removal: {len(df)}")

## 5. Saving Cleaned Data

In [ ]:
if 'df' in locals():
    output_dir = '../../data/cleaned_data/vente/'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_file = 'maison_d_hôte_vente_cleaned.csv'
    output_path = os.path.join(output_dir, output_file)
    df.to_csv(output_path, index=False)
    print(f"Cleaned data saved to {output_path}")